In [1]:
import warnings
warnings.filterwarnings('ignore')

In [3]:
import torch
import torch.nn as nn

In [3]:
class DummyGPTModel(nn.Module):

    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Linear(cfg['vocab_size'], cfg['emb_dim'])
        self.pos_emb = nn.Linear(cfg['context_len'], cfg['emb_dim'])
        self.drop_emb = nn.Dropout(cfg['dropout_rate'])

        self.t_block = nn.Sequential(
            *[DummyTransformerBlock(cfg) for _ in range(cfg['num_layer'])]
        )

        self.final_norm = DummyNormLayer(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg['emd_dim'], cfg['vocab_size'])

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_emb = self.tok_emb(in_idx)
        pos_emb = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_emb + pos_emb
        x = self.drop_emb(x)
        x = self.t_block(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits


class DummyTransformerBlock(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self):
        pass


class DummyNormLayer(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

    def forward(self):
        pass

### Layer normalizatoin

In [5]:
torch.manual_seed(123)

batch_example = torch.randn(2, 5)
batch_example

tensor([[-0.1115,  0.1204, -0.3696, -0.2404, -1.1969],
        [ 0.2093, -0.9724, -0.7550,  0.3239, -0.1085]])

In [6]:
layer = nn.Sequential(nn.Linear(5, 6), nn.ReLU())
out = layer(batch_example)
out

tensor([[0.2260, 0.3470, 0.0000, 0.2216, 0.0000, 0.0000],
        [0.2133, 0.2394, 0.0000, 0.5198, 0.3297, 0.0000]],
       grad_fn=<ReluBackward0>)

In [14]:
mean = out.mean(dim=-1, keepdim=True)
mean

tensor([[0.1324],
        [0.2170]], grad_fn=<MeanBackward1>)

In [15]:
var = out.var(dim=-1, keepdim=True)
var

tensor([[0.0231],
        [0.0398]], grad_fn=<VarBackward0>)

In [20]:
norm = ((out - mean) / torch.sqrt(var)).mean(dim=-1, keepdim=True)
norm

tensor([[-5.9605e-08],
        [ 1.9868e-08]], grad_fn=<MeanBackward1>)

In [21]:
emb_dim = 4

In [40]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True)
        normed = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * normed + self.shift

In [27]:
ln = LayerNorm(6)
normalized = ln(out)


In [30]:
print(normalized.mean(dim=-1, keepdim=True))
print(normalized.var(dim=-1, keepdim=True))

tensor([[-3.9736e-08],
        [ 1.9868e-08]], grad_fn=<MeanBackward1>)
tensor([[0.9996],
        [0.9997]], grad_fn=<VarBackward0>)


### Implemention feed forward network with GELU activation

In [4]:
class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0/torch.pi)) * (x + 0.044715 * torch.pow(x, 3))
        ))

In [5]:
class FeedFrowardNetwork(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg['emb_dim'], 4 * cfg['emb_dim']),
            GELU(),
            nn.Linear(4 * cfg['emb_dim'], cfg['emb_dim'])
        )

    def forward(self, x):
        return self.layers(x)

In [26]:
from main import MultiHeadAttention

In [27]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 1024,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False
}

In [44]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            in_d=cfg['emb_dim'],
            out_d=cfg['emb_dim'] // cfg["n_heads"],
            dropout=cfg['drop_rate'],
            contex_len=cfg['context_length'],
            head=cfg['n_heads'],
            qkv_bias=cfg['qkv_bias']
        )
        self.ff = FeedFrowardNetwork(cfg)
        self.norm1 = LayerNorm(cfg['emb_dim'])
        self.norm2 = LayerNorm(cfg['emb_dim'])
        self.drop_shorcuts = nn.Dropout(cfg['drop_rate'])

    def forward(self, x):
        shortcuts = x
        x = self.norm1(x)
        x = self.att(x)
        x = self.drop_shorcuts(x)
        x = x + shortcuts

        shortcuts = x
        x = self.norm1(x)
        x = self.ff(x)
        x = self.drop_shorcuts(x)
        x = x + shortcuts

        return x

In [45]:
torch.manual_seed(123)

x = torch.rand(2, 3, 768)
block = TransformerBlock(GPT_CONFIG_124M)
output = block(x)

In [46]:
x.shape

torch.Size([2, 3, 768])

In [47]:
output.shape

torch.Size([2, 3, 768])